In [2]:
import dash
from dash import dcc, html
import plotly.express as px
import pandas as pd

# Datos
df = pd.DataFrame({
    "ciudad": ["Madrid", "CDMX", "Buenos Aires"],
    "lat": [40.4168, 19.4326, -34.6037],
    "lon": [-3.7038, -99.1332, -58.3816],
    "ventas": [120, 300, 180]
})

# Mapa
fig = px.scatter_map(
    df,
    lat="lat",
    lon="lon",
    size="ventas",
    hover_name="ciudad",
    zoom=2,
    height=600
)

fig.update_layout(map_style="open-street-map")

# App
app = dash.Dash(__name__)

app.layout = html.Div([
    html.H1("Dashboard de Ventas por Ciudad"),
    dcc.Graph(figure=fig)
])

if __name__ == "__main__":
    app.run(debug=True, port=8051)

In [3]:
import pandas as pd
import plotly.express as px
from dash import Dash, dcc, html, Input, Output


def fetch_fires():
    return pd.DataFrame({
        "lat": [40.4168, 40.42, 40.40, 19.4326, -34.6037],
        "lon": [-3.7038, -3.70, -3.69, -99.1332, -58.3816],
        "date": pd.to_datetime([
            "2024-07-01",
            "2024-07-01",
            "2024-07-02",
            "2024-06-30",
            "2024-06-29"
        ])
    })


fires = fetch_fires().head(10000)

center_lat = fires["lat"].mean()
center_lon = fires["lon"].mean()


def create_map(zoom):

    fig = px.scatter_map(
        fires,
        lat="lat",
        lon="lon",
        zoom=zoom,
        center={"lat": center_lat, "lon": center_lon},
        height=650,
        hover_data=["lat", "lon"]
    )

    fig.update_traces(
        marker=dict(size=8, color="red", opacity=0.6)
    )

    fig.update_layout(
        map_style="open-street-map",
        margin=dict(r=0, l=0, t=40, b=0),
        title=f"Densidad de Incendios (Zoom {zoom})"
    )

    return fig


app = Dash(__name__)

app.layout = html.Div(
    style={"width": "100%", "height": "100vh"},
    children=[
        html.H2("Mapa de Incendios", style={"textAlign": "center"}),

        html.Div(
            style={"width": "300px", "margin": "0 auto"},
            children=[
                html.Label("Nivel de zoom"),
                dcc.Dropdown(
                    id="zoom-selector",
                    options=[
                        {"label": "Mundo", "value": 2},
                        {"label": "Continente", "value": 4},
                        {"label": "Región", "value": 6},
                        {"label": "Zona crítica", "value": 8},
                        {"label": "Detalle máximo", "value": 10},
                    ],
                    value=5,
                    clearable=False
                )
            ]
        ),

        dcc.Graph(
            id="map",
            figure=create_map(5),
            style={"height": "85vh"}
        )
    ]
)


@app.callback(
    Output("map", "figure"),
    Input("zoom-selector", "value")
)
def update_zoom(zoom):
    return create_map(zoom)


if __name__ == "__main__":
    app.run(debug=True, port=8051, use_reloader=False)

In [ ]:
import math
import time
import pandas as pd
import plotly.express as px
from dash import Dash, dcc, html, Input, Output


def fetch_fires():
    return pd.DataFrame({
        "lat": [40.4168, 47.42],
        "lon": [-3.7038, -3.70],
        "date": pd.to_datetime(["2024-07-01", "2024-07-02"]),
        "burned_km": [5, 12]
    })


def fetch_environment(lat, lon, date):
    return {"temp_max": 35}


def build_environmental_df(limit=2, sleep=0):
    fires = fetch_fires().head(limit)
    env_rows = []

    for _, row in fires.iterrows():
        env_rows.append(fetch_environment(row.lat, row.lon, row.date))
        time.sleep(sleep)

    env_df = pd.DataFrame(env_rows)
    return pd.concat([fires.reset_index(drop=True), env_df], axis=1)


df = build_environmental_df(limit=2, sleep=0)

center_lat = df["lat"].mean()
center_lon = df["lon"].mean()


def circle_polygon(lat, lon, radius_km, n_points=40):
    coords = []
    for i in range(n_points + 1):
        angle = 2 * math.pi * i / n_points
        dx = radius_km / 111 * math.cos(angle)
        dy = radius_km / 111 * math.sin(angle)
        coords.append([lon + dx, lat + dy])
    return coords


def create_map(show_radius):

    fig = px.scatter_map(
        df,
        lat="lat",
        lon="lon",
        hover_data=["date", "burned_km"],
        zoom=6,
        center={"lat": center_lat, "lon": center_lon},
        height=600
    )

    fig.update_traces(
        marker=dict(size=10, color="red"),
        name="Inicio incendio"
    )

    fig.update_layout(
        map_style="open-street-map"
    )

    if show_radius:
        layers = []
        for _, row in df.iterrows():
            layers.append({
                "type": "fill",
                "source": {
                    "type": "FeatureCollection",
                    "features": [{
                        "type": "Feature",
                        "geometry": {
                            "type": "Polygon",
                            "coordinates": [[
                                *circle_polygon(
                                    row.lat,
                                    row.lon,
                                    row.burned_km
                                )
                            ]]
                        }
                    }]
                },
                "color": "rgba(255,0,0,0.25)"
            })

        fig.update_layout(map_layers=layers)

    fig.update_layout(
        title="Incendios y radio estimado de área quemada",
        margin=dict(r=0, l=0, t=40, b=0)
    )

    return fig


app = Dash(__name__)

app.layout = html.Div([
    html.H3("Mapa de Incendios (radio en km)"),

    dcc.Checklist(
        id="toggle-radius",
        options=[{"label": " Mostrar área quemada (km)", "value": "show"}],
        value=["show"]
    ),

    dcc.Graph(id="map")
])


@app.callback(
    Output("map", "figure"),
    Input("toggle-radius", "value")
)
def update_map(toggle):
    return create_map(show_radius="show" in toggle)


if __name__ == "__main__":
    app.run(debug=True, port=8051, use_reloader=False)

In [4]:
import math
import pandas as pd
import requests
from dash import Dash, dcc, html, Input, Output
import plotly.express as px


def fetch_fires():
    return pd.DataFrame({
        "lat": [40.4168, 47.42],
        "lon": [-3.7038, -3.70],
        "date": pd.to_datetime(["2024-07-01", "2024-07-02"]),
        "burned_km": [5, 12]
    })


def fetch_environment(lat, lon, date):
    try:
        url = "https://archive-api.open-meteo.com/v1/archive"
        params = {
            "latitude": lat,
            "longitude": lon,
            "start_date": date.strftime("%Y-%m-%d"),
            "end_date": date.strftime("%Y-%m-%d"),
            "daily": ",".join([
                "temperature_2m_mean",
                "temperature_2m_max",
                "temperature_2m_min",
                "relative_humidity_2m_mean",
                "precipitation_sum",
                "wind_speed_10m_max",
                "wind_gusts_10m_max",
                "surface_pressure_mean",
                "cloud_cover_mean",
                "shortwave_radiation_sum",
                "et0_fao_evapotranspiration",
                "sunshine_duration"
            ]),
            "timezone": "UTC"
        }
        r = requests.get(url, params=params)
        r.raise_for_status()
        data = r.json()
        d = data.get("daily", None)

        if d is None:
            raise ValueError("No 'daily' in response")

        return {
            "temp_mean": d["temperature_2m_mean"][0],
            "temp_max": d["temperature_2m_max"][0],
            "temp_min": d["temperature_2m_min"][0],
            "humidity_mean": d["relative_humidity_2m_mean"][0],
            "precipitation": d["precipitation_sum"][0],
            "wind_speed_max": d["wind_speed_10m_max"][0],
            "wind_gusts_max": d["wind_gusts_10m_max"][0],
            "pressure_mean": d["surface_pressure_mean"][0],
            "cloud_cover": d["cloud_cover_mean"][0],
            "radiation": d["shortwave_radiation_sum"][0],
            "evapotranspiration": d["et0_fao_evapotranspiration"][0],
            "sunshine_seconds": d["sunshine_duration"][0]
        }

    except Exception as e:
        print(f" Error al obtener datos ambientales: {e}")
        return {k: None for k in [
            "temp_mean","temp_max","temp_min","humidity_mean","precipitation",
            "wind_speed_max","wind_gusts_max","pressure_mean","cloud_cover",
            "radiation","evapotranspiration","sunshine_seconds"
        ]}


def build_environmental_df(limit=2):
    fires = fetch_fires().head(limit)
    env_rows = []
    for _, row in fires.iterrows():
        env_rows.append(fetch_environment(row.lat, row.lon, row.date))
    env_df = pd.DataFrame(env_rows)
    return pd.concat([fires.reset_index(drop=True), env_df], axis=1)


df = build_environmental_df(limit=2)

center_lat = df["lat"].mean()
center_lon = df["lon"].mean()

ENV_COLS = [
    "temp_mean", "temp_max", "temp_min",
    "humidity_mean", "precipitation",
    "wind_speed_max", "wind_gusts_max",
    "pressure_mean", "cloud_cover",
    "radiation", "evapotranspiration",
    "sunshine_seconds"
]


def circle_polygon(lat, lon, radius_km, n_points=40):
    return [
        [
            lon + (radius_km / 111) * math.cos(2 * math.pi * i / n_points),
            lat + (radius_km / 111) * math.sin(2 * math.pi * i / n_points)
        ]
        for i in range(n_points + 1)
    ]


def create_map(show_radius, show_env):
    hover_data = ["date", "burned_km"]
    if show_env:
        hover_data += ENV_COLS

    fig = px.scatter_map(
        df,
        lat="lat",
        lon="lon",
        hover_data=hover_data,
        zoom=6,
        center={"lat": center_lat, "lon": center_lon},
        height=600
    )

    fig.update_traces(marker=dict(size=10, color="red"), name="Inicio incendio")

    if show_radius:
        fig.update_layout(map_layers=[{
            "type": "fill",
            "source": {
                "type": "FeatureCollection",
                "features": [{
                    "type": "Feature",
                    "geometry": {
                        "type": "Polygon",
                        "coordinates": [[
                            *circle_polygon(
                                df.loc[i, "lat"],
                                df.loc[i, "lon"],
                                df.loc[i, "burned_km"]
                            )
                        ]]
                    }
                }]
            },
            "color": "rgba(255, 0, 0, 0.25)"
        } for i in df.index])

    fig.update_layout(title="Incendios + Datos Ambientales", margin=dict(r=0, l=0, t=40, b=0))
    return fig


app = Dash(__name__)

app.layout = html.Div([
    html.H3("Incendios y Variables Ambientales (2 llamadas API)"),

    dcc.Checklist(
        id="toggle-options",
        options=[
            {"label": " Mostrar área quemada", "value": "radius"},
            {"label": " Mostrar datos ambientales", "value": "env"}
        ],
        value=["radius", "env"],
        inline=True
    ),

    dcc.Graph(id="map")
])


@app.callback(
    Output("map", "figure"),
    Input("toggle-options", "value")
)
def update_map(options):
    return create_map(
        show_radius="radius" in options,
        show_env="env" in options
    )


if __name__ == "__main__":
    app.run(debug=True, port=8051, use_reloader=False)

In [ ]:
import math
import pandas as pd
import requests
from dash import Dash, dcc, html, Input, Output
import plotly.express as px


def fetch_fires(limit=2):
    url = "https://eonet.gsfc.nasa.gov/api/v3/events"
    params = {"status": "open", "category": "wildfires"}
    data = requests.get(url, params=params).json()
    rows = []

    for event in data.get("events", [])[:limit]:
        eid = event.get("id")
        title = event.get("title", "No title")
        for geom in event.get("geometry", []):
            date = geom["date"][:10]
            coords = geom.get("coordinates", [None, None])
            if len(coords) == 2:
                lon, lat = coords
                rows.append([eid, title, date, lat, lon])

    return pd.DataFrame(rows, columns=["id", "title", "date", "lat", "lon"])


def fetch_environment(lat, lon, date):
    try:
        url = "https://archive-api.open-meteo.com/v1/archive"
        params = {
            "latitude": lat,
            "longitude": lon,
            "start_date": date,
            "end_date": date,
            "daily": ",".join([
                "temperature_2m_mean","temperature_2m_max","temperature_2m_min",
                "relative_humidity_2m_mean","precipitation_sum","wind_speed_10m_max",
                "wind_gusts_10m_max","surface_pressure_mean","cloud_cover_mean",
                "shortwave_radiation_sum","et0_fao_evapotranspiration","sunshine_duration"
            ]),
            "timezone": "UTC"
        }
        r = requests.get(url, params=params)
        r.raise_for_status()
        d = r.json().get("daily")
        if not d:
            raise ValueError("No daily data")
        return {
            "temp_mean": d["temperature_2m_mean"][0],
            "temp_max": d["temperature_2m_max"][0],
            "temp_min": d["temperature_2m_min"][0],
            "humidity_mean": d["relative_humidity_2m_mean"][0],
            "precipitation": d["precipitation_sum"][0],
            "wind_speed_max": d["wind_speed_10m_max"][0],
            "wind_gusts_max": d["wind_gusts_10m_max"][0],
            "pressure_mean": d["surface_pressure_mean"][0],
            "cloud_cover": d["cloud_cover_mean"][0],
            "radiation": d["shortwave_radiation_sum"][0],
            "evapotranspiration": d["et0_fao_evapotranspiration"][0],
            "sunshine_seconds": d["sunshine_duration"][0]
        }
    except Exception as e:
        print(f"Error al obtener datos ambientales: {e}")
        return {k: None for k in [
            "temp_mean","temp_max","temp_min","humidity_mean","precipitation",
            "wind_speed_max","wind_gusts_max","pressure_mean","cloud_cover",
            "radiation","evapotranspiration","sunshine_seconds"
        ]}


def build_environmental_df(limit=2):
    fires = fetch_fires(limit=limit)
    env_rows = []
    for _, row in fires.iterrows():
        env_rows.append(fetch_environment(row.lat, row.lon, row.date))
    env_df = pd.DataFrame(env_rows)
    return pd.concat([fires.reset_index(drop=True), env_df], axis=1)


df = build_environmental_df(limit=2)

center_lat = df["lat"].mean()
center_lon = df["lon"].mean()

ENV_COLS = [
    "temp_mean","temp_max","temp_min","humidity_mean","precipitation",
    "wind_speed_max","wind_gusts_max","pressure_mean","cloud_cover",
    "radiation","evapotranspiration","sunshine_seconds"
]


def circle_polygon(lat, lon, radius_km, n_points=40):
    return [
        [
            lon + (radius_km/111) * math.cos(2*math.pi*i/n_points),
            lat + (radius_km/111) * math.sin(2*math.pi*i/n_points)
        ] for i in range(n_points+1)
    ]


def create_map(show_radius, show_env):
    hover_data = ["title","date"]
    if show_env:
        hover_data += ENV_COLS

    fig = px.scatter_map(
        df,
        lat="lat",
        lon="lon",
        hover_data=hover_data,
        zoom=6,
        center={"lat": center_lat, "lon": center_lon},
        height=600
    )

    fig.update_traces(marker=dict(size=10, color="red"), name="Inicio incendio")

    if show_radius:
        fig.update_layout(map_layers=[{
            "type":"fill",
            "source":{
                "type":"FeatureCollection",
                "features":[{
                    "type":"Feature",
                    "geometry":{
                        "type":"Polygon",
                        "coordinates":[
                            *circle_polygon(df.loc[i,"lat"], df.loc[i,"lon"], 10)
                        ]
                    }
                }]
            },
            "color":"rgba(255,0,0,0.25)"
        } for i in df.index])

    fig.update_layout(title="Incendios + Datos Ambientales", margin=dict(r=0,l=0,t=40,b=0))
    return fig


app = Dash(__name__)

app.layout = html.Div([
    html.H3("Incendios NASA EONET + Variables Ambientales"),

    dcc.Checklist(
        id="toggle-options",
        options=[
            {"label":"Mostrar área quemada","value":"radius"},
            {"label":"Mostrar datos ambientales","value":"env"}
        ],
        value=["radius","env"],
        inline=True
    ),

    dcc.Graph(id="map")
])


@app.callback(
    Output("map","figure"),
    Input("toggle-options","value")
)
def update_map(options):
    return create_map(
        show_radius="radius" in options,
        show_env="env" in options
    )


if __name__ == "__main__":
    app.run(debug=True, port=8051, use_reloader=False)